# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset Croissant metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview

Review all available record sets, their `@id`s, fields, and columns present in the dataset. We use the Croissant API to inspect the record set structure.

**Note:** All references to entities use their Croissant `@id`.

In [ ]:
# Inspect available record sets
record_sets = getattr(metadata, 'record_set', [])  # Handles both 'recordSet' and 'record_set'
if not record_sets:
    # Try fallback in case underscore did not work
    record_sets = getattr(metadata, 'recordSet', [])

print("Available record sets and fields:\n")
for rs in record_sets:
    print(f"  RecordSet @id: {rs.id}")
    # List columns/fields
    if hasattr(rs, 'columns'):
        print(f"    Columns:")
        for col in rs.columns:
            print(f"      column @id: {col.id}, name: {col.name}, dataType: {getattr(col, 'data_type', 'N/A')}")
    if hasattr(rs, 'fields'):
        print(f"    Fields:")
        for f in rs.fields:
            print(f"      field @id: {f.id}, name: {f.name}, dataType: {getattr(f, 'data_type', 'N/A')}")
    print()
if not record_sets:
    print("No record sets found in metadata.")

## 3. Data Extraction

Load structured data from each available record set into a DataFrame for analysis.

We use each record set's `@id` to extract tabular data. Columns and fields are referenced by their `@id` as shown above.

Replace `<record_set_id>` (e.g., `'cr:RecordSet/1'`) with a valid value from the overview step when running custom analyses.

In [ ]:
# Collect all record set @id's for extraction
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

# Load records for each record set
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for RecordSet {record_set_id}: {df.shape[0]} rows, {df.shape[1]} columns.")
        else:
            print(f"No records for RecordSet {record_set_id}.")
    except Exception as e:
        print(f"Error loading {record_set_id}: {e}")

# Preview structure of the first available DataFrame
if dataframes:
    example_record_set = next(iter(dataframes.keys()))
    print(f"\nColumns in `{example_record_set}`: \n{dataframes[example_record_set].columns.tolist()}")
    dataframes[example_record_set].head()
else:
    print("No record set data loaded.")

## 4. Exploratory Data Analysis (EDA)

Apply basic data processing: filter records, normalize numeric fields, and group/categorize records.

Please replace `numeric_field_id` and `group_field_id` below with the actual column names (found in the previous cell's output, which are derived from each field's or column's `@id` by Croissant---sometimes these appear as field names, sometimes as `@id`).

In [ ]:
# Change these according to your actual loaded record_set and available numeric fields

# Use the first available DataFrame as example
if dataframes:
    example_record_set = next(iter(dataframes.keys()))
    df = dataframes[example_record_set]
    columns = df.columns.tolist()

    # Guess numeric field (take first float or int if possible)
    numeric_field = None
    for col in df.select_dtypes(include=['number']).columns:
        numeric_field = col
        break
    if not numeric_field:
        print("No numeric fields found for EDA.")
    else:
        print(f"Using field `{numeric_field}` for numeric EDA.")
        threshold = df[numeric_field].mean() if not pd.isna(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.3f}: {len(filtered_df)} rows")

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt grouping by a likely categorical field (first object dtype which isn't the numeric field)
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].dtype == 'object':
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"\nGrouped mean of `{numeric_field}` by `{group_field}`:")
            print(grouped_df.head())
else:
    print("No dataframes available for EDA. Please review earlier steps.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset using `matplotlib`/`seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    # Reuse the EDA dataframe and column selection logic
    example_record_set = next(iter(dataframes.keys()))
    df = dataframes[example_record_set]
    numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
    if numeric_fields:
        field_to_plot = numeric_fields[0]

        plt.figure(figsize=(8, 4))
        sns.histplot(df[field_to_plot].dropna(), kde=True)
        plt.title(f'Distribution of {field_to_plot}')
        plt.xlabel(field_to_plot)
        plt.ylabel('Frequency')
        plt.show()

        # Pair plot if more than one numeric field
        if len(numeric_fields) > 1:
            sns.pairplot(df[numeric_fields[:3]].dropna())
            plt.suptitle("Pairplot of numeric fields", y=1.02)
            plt.show()
    else:
        print("No numeric fields to plot.")
else:
    print("No data to visualize.")

## 6. Conclusion

This notebook demonstrated how to access and process the FAIR² Croissant-format dataset:

- Loading metadata and record sets using `mlcroissant` and referencing all schema entities by `@id`.
- Inspecting available record sets and fields for structured analysis.
- Extracting tabular data for analysis and visualization.
- Filtering, normalizing, and grouping subsets for exploratory data analysis.
- Displaying distributions and relationships in the available data.

Further domain-specific exploration can leverage field descriptions, units, and semantic types, all rigorously mapped in the Croissant schema.